In [1]:
#Train the ML layer. Run once, and again whenever you add labeled examples.
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [2]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODEL_PATH = BASE_DIR / "models" / "scam_model.joblib"

KAGGLE_FILE = DATA_DIR / "fake_job_postings.csv"
OWN_FILE = DATA_DIR / "scam_dataset.csv"
OWN_WEIGHT = 5  # repeat your own labeled examples so local scams count more

print("Working folder:", BASE_DIR)
print("Dataset found:", KAGGLE_FILE.exists())

Working folder: C:\Users\USER\ad classifier
Dataset found: False


In [3]:
df = pd.read_csv("fake_job_postings.csv").fillna("")
df["text"] = (df["title"] + " " + df["company_profile"] + " " + df["description"]
              + " " + df["requirements"] + " " + df["benefits"])
data = df[["text", "fraudulent"]]

print(data.shape)
print(data["fraudulent"].value_counts())

(17880, 2)
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [4]:
if OWN_FILE.exists():
    own = pd.read_csv(OWN_FILE).fillna("")
    own = own.rename(columns={"label": "fraudulent"})[["text", "fraudulent"]]
    print(f"Own examples: {len(own)} ({int(own['fraudulent'].sum())} scam)")
    data = pd.concat([data] + [own] * OWN_WEIGHT, ignore_index=True)
else:
    print("No own dataset yet, training on Kaggle data only.")

No own dataset yet, training on Kaggle data only.


In [5]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,text
0,1,Marketing Intern,"US, NY, New York",Marketing,,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,1,0,Other,Internship,,,Marketing,0,"Marketing Intern We're Food52, and we've creat..."
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,,Marketing and Advertising,Customer Service,0,Customer Service - Cloud Video Production 90 S...
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",,,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,0,1,0,,,,,,0,Commissioning Machinery Assistant (CMA) Valor ...
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,Account Executive - Washington DC Our passion ...
4,5,Bill Review Manager,"US, FL, Fort Worth",,,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,Bill Review Manager SpotSource Solutions LLC i...


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["fraudulent"], test_size=0.2,
    stratify=data["fraudulent"], random_state=42,
)

model = make_pipeline(
    TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2)),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3403
           1       0.74      0.90      0.81       173

    accuracy                           0.98      3576
   macro avg       0.87      0.94      0.90      3576
weighted avg       0.98      0.98      0.98      3576

Confusion matrix (rows = actual, columns = predicted):
[[3349   54]
 [  17  156]]


In [11]:
import seaborn as sns
cm = "Confusion matrix (rows = actual, columns = predicted)"
sns.heatmap(cm, annot=True, fmt='d', xticklabels=[0, 1], yticklabels=[0, 1], cmap='bwr')
plt.title("Confusion Matrix")
plt.xlabel("predicted label")
plt.ylabel("actual label")
plt.show()

ValueError: Must pass 2-d input. shape=()

In [7]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH)

Saved: C:\Users\USER\ad classifier\models\scam_model.joblib


In [8]:
tests = {
    "scam": "Earn $200/day liking videos. Pay a registration fee and send your BVN. WhatsApp only.",
    "legit": "Junior Software Engineer at Kano Tech Ltd. Degree in Computer Science required. Send CV to careers@kanotech.com.",
}
for name, text in tests.items():
    print(name, "->", round(model.predict_proba([text])[0][1] * 100), "/100")

scam -> 79 /100
legit -> 39 /100
